In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

## 1. Generate Customers Dataset

In [2]:
n_customers = 5000

# Generate customer IDs
customer_ids = list(range(1, n_customers + 1))

# Generate signup dates (between 2 years ago and 6 months ago)
start_date = datetime(2022, 1, 1)
end_date = datetime(2024, 6, 30)
date_range = (end_date - start_date).days
signup_dates = [
    start_date + timedelta(days=random.randint(0, date_range))
    for _ in range(n_customers)
]

# Intentionally use DD/MM/YYYY format for staging practice
signup_dates_str = [d.strftime("%d/%m/%Y") for d in signup_dates]

# Generate countries with inconsistent naming
countries_pool = [
    "USA",
    "United States",
    "US",  # Intentionally inconsistent
    "UK",
    "United Kingdom",
    "Britain",  # Intentionally inconsistent
    "Germany",
    "Spain",
    "France",
    "Italy",
    "Canada",
    "Australia",
    "Brazil",
    "Mexico",
]
countries = np.random.choice(countries_pool, n_customers)

# Generate ages (18-70) with some missing values
ages = np.random.randint(18, 71, n_customers).astype(float)
# Introduce ~5% missing values
missing_age_idx = np.random.choice(
    n_customers, size=int(n_customers * 0.05), replace=False
)
ages[missing_age_idx] = np.nan

# Generate gender with some missing values
genders = np.random.choice(["M", "F", "Other"], n_customers, p=[0.48, 0.48, 0.04])
# Introduce ~3% missing values
missing_gender_idx = np.random.choice(
    n_customers, size=int(n_customers * 0.03), replace=False
)
genders = genders.astype(object)
genders[missing_gender_idx] = np.nan

# Generate subscription tiers
subscription_tiers = np.random.choice(
    ["basic", "Premium", "ENTERPRISE"], n_customers, p=[0.6, 0.3, 0.1]
)

# Generate monthly fees based on tier
monthly_fees = []
for tier in subscription_tiers:
    if tier.lower() == "basic":
        monthly_fees.append(round(np.random.uniform(9.99, 14.99), 2))
    elif tier.lower() == "premium":
        monthly_fees.append(round(np.random.uniform(29.99, 39.99), 2))
    else:  # Enterprise
        monthly_fees.append(round(np.random.uniform(99.99, 199.99), 2))

# Create DataFrame
customers_df = pd.DataFrame(
    {
        "customer_id": customer_ids,
        "signup_date": signup_dates_str,
        "country": countries,
        "age": ages,
        "gender": genders,
        "subscription_tier": subscription_tiers,
        "monthly_fee": monthly_fees,
    }
)

print(f"Created customers dataset: {customers_df.shape}")
print(f"Missing ages: {customers_df['age'].isna().sum()}")
print(f"Missing genders: {customers_df['gender'].isna().sum()}")
customers_df.head(10)

Created customers dataset: (5000, 7)
Missing ages: 250
Missing genders: 150


,customer_id,signup_date,country,age,gender,subscription_tier,monthly_fee
0,1,17/10/2023,Germany,62.0,F,basic,13.00
1,2,25/04/2022,UK,NaN,M,Premium,30.48
2,3,26/01/2022,Brazil,53.0,NaN,basic,11.59
3,4,30/01/2024,Canada,26.0,M,basic,11.97
4,5,09/10/2022,Spain,44.0,F,basic,11.23
5,6,08/09/2022,Brazil,51.0,F,basic,10.18
6,7,17/08/2022,United Kingdom,68.0,M,ENTERPRISE,131.87
7,8,23/05/2022,Germany,18.0,M,basic,13.95
8,9,25/01/2024,Italy,NaN,F,basic,11.79
9,10,15/04/2022,US,42.0,M,ENTERPRISE,132.33


## 2. Generate Usage Logs Dataset

In [3]:
# Generate usage logs for the last 90 days before observation date
observation_date = datetime(2024, 12, 31)
log_start_date = observation_date - timedelta(days=89)  # 90 days total

usage_logs = []

for customer_id in customer_ids:
    # Some customers are more active than others
    activity_level = np.random.choice(["low", "medium", "high"], p=[0.3, 0.5, 0.2])

    if activity_level == "low":
        days_to_log = np.random.randint(5, 30)  # Low activity
    elif activity_level == "medium":
        days_to_log = np.random.randint(30, 60)  # Medium activity
    else:
        days_to_log = np.random.randint(60, 90)  # High activity

    # Select random days within the 90-day period
    active_days = np.random.choice(90, days_to_log, replace=False)

    for day_offset in active_days:
        log_date = log_start_date + timedelta(days=int(day_offset))

        sessions = np.random.randint(1, 10)
        duration = round(
            np.random.gamma(2, 15), 2
        )  # Gamma distribution for realistic usage

        # Introduce some negative duration errors (~1%)
        if np.random.random() < 0.01:
            duration = -duration

        features_used = np.random.randint(1, 15)
        errors = np.random.poisson(0.5)  # Poisson distribution for errors

        usage_logs.append(
            {
                "customer_id": customer_id,
                "log_date": log_date.strftime("%Y-%m-%d"),
                "sessions": sessions,
                "duration_minutes": duration,
                "features_used": features_used,
                "errors_encountered": errors,
            }
        )

usage_logs_df = pd.DataFrame(usage_logs)

print(f"Created usage logs dataset: {usage_logs_df.shape}")
print(f"Negative durations: {(usage_logs_df['duration_minutes'] < 0).sum()}")
usage_logs_df.head(10)

Created usage logs dataset: (212936, 6)
Negative durations: 2128


,customer_id,log_date,sessions,duration_minutes,features_used,errors_encountered
0,1,2024-12-08,9,44.92,6,0
1,1,2024-11-19,1,49.53,1,0
2,1,2024-12-12,4,7.12,1,0
3,1,2024-11-14,7,44.35,6,0
4,1,2024-11-15,2,29.12,12,1
5,1,2024-11-28,7,6.14,3,1
6,1,2024-11-13,6,47.63,14,1
7,1,2024-12-23,9,28.55,13,0
8,1,2024-10-26,5,19.75,7,0
9,1,2024-12-18,5,36.66,5,1


## 3. Generate Support Tickets Dataset

In [4]:
# Not all customers create tickets
# About 40% of customers will have at least one ticket
customers_with_tickets = np.random.choice(
    customer_ids, size=int(n_customers * 0.4), replace=False
)

tickets = []
ticket_id = 1

for customer_id in customers_with_tickets:
    # Each customer with tickets has 1-5 tickets
    n_tickets = np.random.randint(1, 6)

    for _ in range(n_tickets):
        # Create ticket sometime in the last 365 days
        created_date = observation_date - timedelta(days=np.random.randint(1, 366))
        created_str = created_date.strftime("%Y/%m/%d %H:%M:%S")

        # 80% of tickets are resolved
        is_resolved = np.random.random() < 0.8

        if is_resolved:
            # Resolution takes 1-72 hours
            resolution_hours = np.random.gamma(2, 8)  # Gamma distribution
            resolved_date = created_date + timedelta(hours=resolution_hours)
            resolved_str = resolved_date.strftime("%Y/%m/%d %H:%M:%S")
        else:
            resolved_str = np.nan

        category = np.random.choice(
            ["Technical", "Billing", "General"], p=[0.5, 0.3, 0.2]
        )
        priority = np.random.choice(["Low", "Medium", "High"], p=[0.5, 0.35, 0.15])

        # Satisfaction score only for ~60% of resolved tickets
        if is_resolved and np.random.random() < 0.6:
            # Higher priority tickets tend to have lower satisfaction
            if priority == "High":
                satisfaction = np.random.choice(
                    [1, 2, 3, 4, 5], p=[0.1, 0.2, 0.3, 0.3, 0.1]
                )
            elif priority == "Medium":
                satisfaction = np.random.choice(
                    [1, 2, 3, 4, 5], p=[0.05, 0.1, 0.2, 0.4, 0.25]
                )
            else:
                satisfaction = np.random.choice(
                    [1, 2, 3, 4, 5], p=[0.02, 0.05, 0.13, 0.4, 0.4]
                )
        else:
            satisfaction = np.nan

        tickets.append(
            {
                "ticket_id": ticket_id,
                "customer_id": customer_id,
                "created_date": created_str,
                "resolved_date": resolved_str,
                "category": category,
                "priority": priority,
                "satisfaction_score": satisfaction,
            }
        )

        ticket_id += 1

support_tickets_df = pd.DataFrame(tickets)

print(f"Created support tickets dataset: {support_tickets_df.shape}")
print(f"Unresolved tickets: {support_tickets_df['resolved_date'].isna().sum()}")
print(
    f"Missing satisfaction scores: {support_tickets_df['satisfaction_score'].isna().sum()}"
)
support_tickets_df.head(10)

Created support tickets dataset: (5944, 7)
Unresolved tickets: 1204
Missing satisfaction scores: 3093


,ticket_id,customer_id,created_date,resolved_date,category,priority,satisfaction_score
0,1,4110,2024/03/29 00:00:00,2024/03/29 08:51:44,Technical,Medium,NaN
1,2,4110,2024/03/04 00:00:00,2024/03/04 12:08:22,Billing,Low,NaN
2,3,4110,2024/03/27 00:00:00,2024/03/27 01:09:27,Technical,Medium,5.0
3,4,2990,2024/12/22 00:00:00,2024/12/23 03:33:56,Technical,Low,NaN
4,5,3341,2024/01/03 00:00:00,2024/01/05 16:12:29,Technical,High,NaN
5,6,3341,2024/10/01 00:00:00,2024/10/01 04:40:00,Technical,Medium,NaN
6,7,3341,2024/06/02 00:00:00,2024/06/02 13:02:55,General,Low,1.0
7,8,3341,2024/09/20 00:00:00,NaN,Technical,High,NaN
8,9,3341,2024/03/29 00:00:00,NaN,Technical,Medium,NaN
9,10,2669,2024/05/06 00:00:00,2024/05/06 09:53:15,Billing,Low,NaN


## 4. Generate Churn Dataset

In [5]:
# Generate churn labels
# Churn rate ~25%
churned_customers = np.random.choice(
    customer_ids, size=int(n_customers * 0.25), replace=False
)

churn_data = []

for customer_id in customer_ids:
    if customer_id in churned_customers:
        # Churn date is sometime in the last 90 days before observation
        days_ago = np.random.randint(1, 91)
        churn_date = observation_date - timedelta(days=days_ago)
        churn_date_str = churn_date.strftime("%Y-%m-%d")
        churned = 1
    else:
        churn_date_str = np.nan
        churned = 0

    churn_data.append(
        {
            "customer_id": customer_id,
            "churn_date": churn_date_str,
            "churned": churned,
            "observation_date": observation_date.strftime("%Y-%m-%d"),
        }
    )

churn_df = pd.DataFrame(churn_data)

print(f"Created churn dataset: {churn_df.shape}")
print(f"Churn rate: {churn_df['churned'].mean():.2%}")
churn_df.head(10)

Created churn dataset: (5000, 4)
Churn rate: 25.00%


,customer_id,churn_date,churned,observation_date
0,1,NaN,0,2024-12-31
1,2,NaN,0,2024-12-31
2,3,NaN,0,2024-12-31
3,4,NaN,0,2024-12-31
4,5,NaN,0,2024-12-31
5,6,NaN,0,2024-12-31
6,7,2024-12-04,1,2024-12-31
7,8,NaN,0,2024-12-31
8,9,NaN,0,2024-12-31
9,10,NaN,0,2024-12-31


## 5. Save All Datasets to CSV

In [6]:
# Save to raw folder
customers_df.to_csv("../raw/customers.csv", index=False)
usage_logs_df.to_csv("../raw/usage_logs.csv", index=False)
support_tickets_df.to_csv("../raw/support_tickets.csv", index=False)
churn_df.to_csv("../raw/churn.csv", index=False)

print("✅ All datasets saved to ../raw/ folder")
print(f"   - customers.csv: {len(customers_df):,} rows")
print(f"   - usage_logs.csv: {len(usage_logs_df):,} rows")
print(f"   - support_tickets.csv: {len(support_tickets_df):,} rows")
print(f"   - churn.csv: {len(churn_df):,} rows")

✅ All datasets saved to ../raw/ folder
   - customers.csv: 5,000 rows
   - usage_logs.csv: 212,936 rows
   - support_tickets.csv: 5,944 rows
   - churn.csv: 5,000 rows


## Summary Statistics

In [7]:
print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"\nTotal Customers: {n_customers:,}")
print(f"Observation Date: {observation_date.strftime('%Y-%m-%d')}")
print(f"\nChurn Rate: {churn_df['churned'].mean():.2%}")
print(
    f"Customers with Tickets: {len(customers_with_tickets):,} ({len(customers_with_tickets)/n_customers:.1%})"
)
print(f"Total Usage Log Entries: {len(usage_logs_df):,}")
print(f"Total Support Tickets: {len(support_tickets_df):,}")
print(f"\nData Quality Issues Introduced:")
print(f"  - Inconsistent date formats: ✓")
print(f"  - Missing values in age: {customers_df['age'].isna().sum()}")
print(f"  - Missing values in gender: {customers_df['gender'].isna().sum()}")
print(f"  - Negative duration values: {(usage_logs_df['duration_minutes'] < 0).sum()}")
print(f"  - Inconsistent country names: ✓")
print(f"  - Mixed case subscription tiers: ✓")
print("=" * 60)

DATASET SUMMARY

Total Customers: 5,000
Observation Date: 2024-12-31

Churn Rate: 25.00%
Customers with Tickets: 2,000 (40.0%)
Total Usage Log Entries: 212,936
Total Support Tickets: 5,944

Data Quality Issues Introduced:
  - Inconsistent date formats: ✓
  - Missing values in age: 250
  - Missing values in gender: 150
  - Negative duration values: 2128
  - Inconsistent country names: ✓
  - Mixed case subscription tiers: ✓
